# Prática 5 — Filtro ativo Butterworth passa-baixa de 4ª ordem

**Análise teórica e dimensionamento em Python (UFPE — Circuitos II)**  
**Especificações:** $f_c=50\,\mathrm{Hz}$, $n=4$, ganho total $K=4$ e $C_{\max}=100\,\mathrm{nF}$.

Este notebook reproduz a **parte teórica** da resolução: análise nodal, impedância de entrada, protótipo Butterworth, polos, escalonamento, associação de capacitores disponíveis, funções de transferência, fator de qualidade, resposta analítica em frequência e Bode.

O dimensionamento físico emprega **somente capacitores de 10, 22, 47, 68 ou 100 nF**, conforme disponibilidade informada. Na combinação proposta, são utilizados valores de **22, 47 e 100 nF**. O número necessário de cada um aparece adiante.

> **Escopo:** não inclui a análise do relatório de Felipe, nem resultados experimentais não fornecidos. O ganho $K=4$ é realizado por um **estágio amplificador independente** dos dois Sallen-Key de ganho unitário.

## 1. Bibliotecas e especificações

- `sympy`: desenvolvimento algébrico;
- `numpy` e `scipy`: avaliação numérica e busca de frequências;
- `matplotlib`: diagramas de Bode;
- `pandas`: tabelas dos resultados.

As quantidades são armazenadas em unidades SI (Hz, $\Omega$, F e rad/s).

In [1]:
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from IPython.display import display, Math

sp.init_printing()

ORDEM = 4
FC = 50.0                        # Hz
WC = 2 * np.pi * FC              # rad/s
GANHO = 4.0                      # V/V, estágio independente
C_MAX = 100e-9                   # F
R_MOD = 82e3                     # ohms por resistor, nos dois filtros
VIN_PP = 5.0                     # Vpp: especificação do ensaio do roteiro

print(f'Ordem = {ORDEM}')
print(f'f_c = {FC:.3f} Hz; omega_c = {WC:.6f} rad/s')
print(f'K = {GANHO:.1f} V/V; C_max = {C_MAX*1e9:.1f} nF')
print(f'Resistores adotados = {R_MOD/1e3:.0f} kΩ')

ModuleNotFoundError: No module named 'numpy'

## 2. Análise nodal do Sallen–Key

Na topologia do roteiro, $C_1$ liga o nó intermediário $V_a$ à saída do amplificador operacional; $C_2$ liga o nó da entrada não inversora ao terra. O amplificador é seguidor de tensão e, idealmente, $V_o=V_+=V_-$.

$$
\begin{aligned}
\text{Nó }V_a:\quad &\frac{V_a-V_i}{R_1}+\frac{V_a-V_o}{R_2}+sC_1(V_a-V_o)=0,\\
\text{Nó }V_+:\quad &\frac{V_o-V_a}{R_2}+sC_2V_o=0.
\end{aligned}
$$

Da segunda equação, $V_a=V_o(1+sR_2C_2)$. A célula a seguir substitui essa expressão na primeira e obtém $H_m(s)=V_o/V_i$ **simbolicamente**.

In [ ]:
s = sp.symbols('s')
R1, R2, C1, C2 = sp.symbols('R_1 R_2 C_1 C_2', positive=True)
Vi, Va, Vo = sp.symbols('V_i V_a V_o')

no_mais = sp.Eq((Vo - Va)/R2 + s*C2*Vo, 0)
va_sol = sp.solve(no_mais, Va)[0]
no_va = sp.Eq((Va - Vi)/R1 + (Va - Vo)/R2 + s*C1*(Va - Vo), 0)
vi_sol = sp.factor(sp.solve(no_va.subs(Va, va_sol), Vi)[0])
H_m = sp.factor(Vo / vi_sol)
H_esperado = 1/(R1*R2*C1*C2*s**2+C2*(R1+R2)*s+1)
assert sp.simplify(H_m-H_esperado) == 0

print('Tensão intermediária:')
display(sp.Eq(Va, va_sol))
print('Tensão de entrada em função da saída:')
display(sp.Eq(Vi, vi_sol))
print('Função de transferência geral:')
display(sp.Eq(sp.Symbol('H_m'), H_m))
print('Resistores iguais R1=R2=R:')
R = sp.symbols('R', positive=True)
H_m_iguais = sp.factor(H_m.subs({R1:R,R2:R}))
display(sp.Eq(sp.Symbol('H_m'), H_m_iguais))

### 2.1. Impedância de entrada

$$
I_i=\frac{V_i-V_a}{R_1},\qquad Z_{in}(s)=\frac{V_i}{I_i}.
$$

A dedução abaixo utiliza as mesmas equações nodais e mantém os resistores inicialmente diferentes:

In [ ]:
Ii = sp.factor((vi_sol-va_sol)/R1)
Zin = sp.factor(vi_sol/Ii)
Zin_ref = (R1*R2*C1*C2*s**2+C2*(R1+R2)*s+1)/(s*C2*(1+s*R2*C1))
assert sp.simplify(Zin-Zin_ref) == 0

display(sp.Eq(sp.Symbol('I_i'), Ii))
display(sp.Eq(sp.Symbol('Z_in'), Zin))
print('Limite de baixa frequência da impedância de entrada:')
display(sp.limit(Zin,s,0,dir='+'))

## 3. Protótipo Butterworth normalizado

Para $n=4$, com $\omega_c=1\,\mathrm{rad/s}$ e ganho unitário:

$$
|H_p(j\omega)|^2=\frac{1}{1+\omega^8}.
$$

Os polos são as quatro raízes estáveis da equação $s^8=-1$:

$$
p_k=e^{j(2k+1)\pi/8},\qquad \theta\in\left\{\frac{5\pi}8,\frac{7\pi}8,\frac{9\pi}8,\frac{11\pi}8\right\}.
$$

Os fatores de segunda ordem são:

$$
D_1(s)=s^2+b_1s+1,\qquad D_2(s)=s^2+b_2s+1,
$$

com $b_1=\sqrt{2-\sqrt2}$ e $b_2=\sqrt{2+\sqrt2}$.

In [ ]:
b1_sim = sp.sqrt(2-sp.sqrt(2))
b2_sim = sp.sqrt(2+sp.sqrt(2))
b1, b2 = float(b1_sim), float(b2_sim)

angulos = [5*sp.pi/8,7*sp.pi/8,9*sp.pi/8,11*sp.pi/8]
polos = [complex(sp.N(sp.exp(sp.I*ang),16)) for ang in angulos]
for k,p in enumerate(polos,1):
    print(f'p_{k} = {p.real:+.9f} {p.imag:+.9f}j')
print(f'b_1 = {b1:.9f}; b_2 = {b2:.9f}')
print(f'Q_1 ideal = {1/b1:.9f}; Q_2 ideal = {1/b2:.9f}')

D1_norm = s**2+b1_sim*s+1
D2_norm = s**2+b2_sim*s+1
D_norm = sp.expand(D1_norm*D2_norm)
Hp = 1/(D1_norm*D2_norm)

print('Protótipo fatorado:')
display(sp.Eq(sp.Symbol('H_p'), Hp))
print('Denominador expandido:')
display(sp.Eq(sp.Symbol('D_p'), D_norm))
print('Coeficientes numéricos em ordem descendente:')
for grau,coef in zip(range(4,-1,-1),sp.Poly(D_norm,s).all_coeffs()):
    print(f's^{grau}: {float(coef):.9f}')

### 3.1. Prova algébrica da resposta Butterworth

Fazendo $x=\omega/\omega_c$:

$$
H(j\omega)=\frac{K}{(1-x^2+jb_1x)(1-x^2+jb_2x)}.
$$

Logo,

$$
|H(j\omega)|^2=\frac{K^2}{[(1-x^2)^2+b_1^2x^2][(1-x^2)^2+b_2^2x^2]}.
$$

O produto dos fatores reais deve simplificar exatamente para $1+x^8$.

In [ ]:
x = sp.symbols('x',real=True)
F1 = (1-x**2)**2+b1_sim**2*x**2
F2 = (1-x**2)**2+b2_sim**2*x**2
produto = sp.simplify(sp.expand(F1*F2))
assert sp.simplify(produto-(1+x**8)) == 0

display(sp.Eq(sp.Symbol('F_1'),sp.expand(F1)))
display(sp.Eq(sp.Symbol('F_2'),sp.expand(F2)))
display(sp.Eq(sp.Symbol('F_1F_2'),produto))
print('Magnitude ideal = K / sqrt(1+(f/FC)^8).')

## 4. Projeto dos componentes do protótipo

Para $R_1=R_2=R$:

$$
H_m(s)=\frac{1}{R^2C_1C_2s^2+2RC_2s+1}.
$$

Igualando coeficientes a $1/(s^2+bs+1)$:

$$
R^2C_1C_2=1,\qquad 2RC_2=b.
$$

Assim,

$$\boxed{C_1=\frac{2}{Rb},\qquad C_2=\frac{b}{2R}}$$

Calcularemos para o protótipo com $R=1\,\Omega$.

In [ ]:
R_PROTO = 1.0  # ohm
proto = []
for nome,b in [('Módulo 1',b1),('Módulo 2',b2)]:
    c1 = 2/(R_PROTO*b)
    c2 = b/(2*R_PROTO)
    proto.append((c1,c2))
    print(f'{nome}: C1 = {c1:.9f} F; C2 = {c2:.9f} F')

assert np.isclose(max(np.array(proto).ravel()),proto[0][0])

## 5. Escalonamento de frequência e impedância

$$k_f=\omega_c=2\pi f_c,\qquad R'=k_zR,\qquad C'=\frac{C}{k_fk_z}.$$

Impõe-se que a maior capacitância física seja $100\,\mathrm{nF}$:

$$
k_z=\frac{C_{\max,p}}{k_f C_{\max,\mathrm{físico}}}.
$$

O cálculo usa todos os valores do protótipo e não arredonda os valores intermediários.

In [ ]:
KF = WC
C_MAX_PROTO = max(c for secao in proto for c in secao)
KZ = C_MAX_PROTO / (KF*C_MAX)
R_TEORICO = KZ*R_PROTO

teorico=[]
for nome,(c1p,c2p) in zip(['Módulo 1','Módulo 2'],proto):
    teorico.append((c1p/(KF*KZ),c2p/(KF*KZ)))

print(f'k_f = {KF:.9f}')
print(f'k_z = {KZ:.6f}')
print(f'R teórico = {R_TEORICO/1e3:.6f} kΩ')
for nome,(c1,c2) in zip(['Módulo 1','Módulo 2'],teorico):
    print(f'{nome}: C1={c1*1e9:.6f} nF, C2={c2*1e9:.6f} nF')

assert np.isclose(max(c for etapa in teorico for c in etapa),C_MAX)

## 6. Dimensionamento usando os capacitores disponíveis

**Posições no circuito:** $C_1$ é o capacitor de realimentação entre o ponto médio dos resistores e a saída; $C_2$ é o capacitor conectado ao terra junto à entrada não inversora.

O valor da associação em série é

$$C_s=\frac{C_aC_b}{C_a+C_b},$$

em paralelo:

$$C_p=C_a+C_b.$$

**Associações escolhidas:**

| Estágio | Posição | Associação | Fórmula |
|---|---|---|---|
| 1 | $C_1$ | 100 nF | $100$ |
| 1 | $C_2$ | 22 nF em série com 47 nF | $22\cdot47/(22+47)$ |
| 2 | $C_1$ | 22 nF em paralelo com 22 nF | $22+22$ |
| 2 | $C_2$ | 22 nF em paralelo com (22 nF em série com 47 nF) | $22+22\cdot47/(22+47)$ |

**Inventário mínimo dessa alternativa:** 5 unidades de 22 nF, 2 unidades de 47 nF e 1 unidade de 100 nF. Os valores de 10 e 68 nF não são necessários nessa versão. O notebook **não presume** que você possua essas quantidades.

In [ ]:
def serie(*caps):
    """Capacitância equivalente de dois ou mais capacitores, em farads."""
    caps=np.asarray(caps,dtype=float)
    if np.any(caps <= 0): raise ValueError('Capacitâncias devem ser positivas.')
    return 1.0/np.sum(1.0/caps)

def paralelo(*caps):
    """Capacitância equivalente de dois ou mais capacitores, em farads."""
    caps=np.asarray(caps,dtype=float)
    if np.any(caps <= 0): raise ValueError('Capacitâncias devem ser positivas.')
    return float(np.sum(caps))

C1_A = 100e-9
C2_A = serie(22e-9,47e-9)
C1_B = paralelo(22e-9,22e-9)
C2_B = paralelo(22e-9, serie(22e-9,47e-9))

etapas=[('Módulo 1',R_MOD,R_MOD,C1_A,C2_A),
        ('Módulo 2',R_MOD,R_MOD,C1_B,C2_B)]

comparacao=[]
for (nome,r1,r2,c1,c2),(ct1,ct2) in zip(etapas,teorico):
    comparacao.append({
        'Etapa':nome,'R1 [kΩ]':r1/1e3,'R2 [kΩ]':r2/1e3,
        'C1 teórico [nF]':ct1*1e9,'C1 adotado [nF]':c1*1e9,
        'C2 teórico [nF]':ct2*1e9,'C2 adotado [nF]':c2*1e9,
    })

display(pd.DataFrame(comparacao).round(5))
assert all(c1<=C_MAX*(1+1e-12) and c2<=C_MAX*(1+1e-12) for _,_,_,c1,c2 in etapas)
print('Associados: C2 módulo 1 =',round(C2_A*1e9,6),'nF; C2 módulo 2 =',round(C2_B*1e9,6),'nF')

## 7. Funções de transferência dos dois módulos físicos

Para resistores possivelmente distintos, por generalidade:

$$H_i(s)=\frac{1}{a_is^2+d_is+1},\qquad a_i=R_{1i}R_{2i}C_{1i}C_{2i},\qquad d_i=C_{2i}(R_{1i}+R_{2i}).$$

Os dois seguidores têm ganho unitário. Para obter $K=4$, adiciona-se um **terceiro estágio não inversor**, idealmente independente:

$$K=1+\frac{R_f}{R_g}=4\;\Rightarrow\; R_g=10\,\mathrm{k\Omega},\quad R_f=30\,\mathrm{k\Omega}.$$

Logo, $H(s)=4H_1(s)H_2(s)$.

In [ ]:
coeffs=[]
for nome,r1,r2,c1,c2 in etapas:
    a=r1*r2*c1*c2
    d=c2*(r1+r2)
    coeffs.append((a,d))
    print(f'{nome}: a = {a:.12g} s²; d = {d:.12g} s')
    display(sp.Eq(sp.Symbol('H_'+nome.replace(' ','_')),
                     1/(sp.Float(a)*s**2+sp.Float(d)*s+1)))

(a1,d1),(a2,d2)=coeffs
polinomio=[a1*a2,a1*d2+a2*d1,a1+a2+d1*d2,d1+d2,1.0]
print('\nCoeficientes D(s) = a4*s^4+a3*s^3+a2*s²+a1*s+1:')
for potencia,coef in zip(range(4,-1,-1),polinomio):
    print(f's^{potencia}: {coef:.12g}')
print('\nH_total(s) = 4 / D(s).')

RG = 10e3
RF = 30e3
assert np.isclose(1+RF/RG, GANHO)

## 8. Frequência natural e fator de qualidade de cada módulo

Com a forma canônica

$$H_i(s)=\frac{\omega_{0i}^2}{s^2+\frac{\omega_{0i}}{Q_i}s+\omega_{0i}^2},$$

identificamos:

$$\omega_{0i}=\frac{1}{\sqrt{a_i}},\quad f_{0i}=\frac{1}{2\pi\sqrt{a_i}},\quad Q_i=\frac{\sqrt{a_i}}{d_i}.$$

Para $R_1=R_2=R$, também vale $Q=\frac12\sqrt{C_1/C_2}$.

In [ ]:
qualidade=[]
for (nome,r1,r2,c1,c2),(a,d),b_ideal in zip(etapas,coeffs,[b1,b2]):
    f0=1/(2*np.pi*np.sqrt(a))
    q=np.sqrt(a)/d
    qualidade.append({'Etapa':nome,'f0 [Hz]':f0,'Q calculado':q,'Q Butterworth ideal':1/b_ideal,
                      'Diferença Q [%]':100*(q/(1/b_ideal)-1)})
display(pd.DataFrame(qualidade).round(6))

## 9. Resposta analítica ideal e do circuito com capacitores disponíveis

Para o filtro ideal de 4ª ordem:

$$|H_{\mathrm{ideal}}(j2\pi f)|=\frac{K}{\sqrt{1+(f/f_c)^8}}.$$

$$G_{\mathrm{ideal},\mathrm{dB}}(f)=20\log_{10}(K)-10\log_{10}\left[1+(f/f_c)^8\right].$$

Para o filtro efetivamente dimensionado:

$$H_{\mathrm{adotado}}(j\omega)=\frac{K}{[1-a_1\omega^2+jd_1\omega][1-a_2\omega^2+jd_2\omega]}.$$

$$\boxed{\ |H_{\mathrm{adotado}}(j\omega)|=\frac{K}{\sqrt{[(1-a_1\omega^2)^2+(d_1\omega)^2][(1-a_2\omega^2)^2+(d_2\omega)^2]}}\ }.$$

Os cálculos seguintes usam a **resposta complexa**, permitindo obter também a fase e localizar o corte real.

In [ ]:
def h_ideal(f, ganho=GANHO):
    f=np.asarray(f,dtype=float)
    x=f/FC
    return ganho / ((1-x**2+1j*b1*x)*(1-x**2+1j*b2*x))

def h_adotado(f, ganho=GANHO):
    f=np.asarray(f,dtype=float)
    w=2*np.pi*f
    return ganho / ((1-a1*w**2+1j*d1*w)*(1-a2*w**2+1j*d2*w))

def db(z):
    return 20*np.log10(np.abs(z))

# Identidade analítica Butterworth: magnitude obtida pela forma fatorada
# precisa coincidir com K/sqrt(1+(f/FC)^8).
f_teste=np.geomspace(0.01,10000,1000)
mag_fechada=GANHO/np.sqrt(1+(f_teste/FC)**8)
assert np.allclose(np.abs(h_ideal(f_teste)),mag_fechada,rtol=1e-12)

print(f'Ganho na banda de passagem: {GANHO:.4f} V/V = {20*np.log10(GANHO):.4f} dB')
print(f'Magnitude ideal a 50 Hz: {np.abs(h_ideal(50)):.6f} V/V')
print(f'Atenuação ideal relativa a 50 Hz: {db(h_ideal(50))-20*np.log10(GANHO):.6f} dB')

## 10. Frequências de corte e atenuações de −10 dB e −34 dB

Os níveis são **relativos à banda de passagem**, como solicitado no roteiro. Portanto, a condição é:

$$20\log_{10}\left(\frac{|H(j2\pi f)|}{K}\right)=A_{\mathrm{rel}}.$$

Para o Butterworth ideal é possível isolar $f$:

$$\boxed{f= f_c\left(10^{-A_{\mathrm{rel}}/10}-1\right)^{1/8}}.$$

Para a montagem adotada, com $y=\omega^2$, resolvemos numericamente a equação

$$[(1-a_1 y)^2+d_1^2y][(1-a_2 y)^2+d_2^2y]=10^{-A_{\mathrm{rel}}/10}.$$

Em particular, para $-3{,}0103\,\mathrm{dB}$ o lado direito é 2, uma equação de quarto grau em $y$.

In [ ]:
def freq_ideal_atenuacao(atten_db):
    return FC*(10**(-atten_db/10)-1)**(1/8)

def freq_adotada_atenuacao(atten_db):
    alvo=10**(atten_db/20)           # |H|/K
    def func(f):
        return np.abs(h_adotado(f))/GANHO-alvo
    return brentq(func,1e-8,1e5,xtol=1e-11)

atenuacoes=[-10*np.log10(2),-10.0,-34.0]
nomes=['Corte (−3,0103 dB)','−10 dB','−34 dB']
rows=[]
for nome,a_db in zip(nomes,atenuacoes):
    f_ideal=freq_ideal_atenuacao(a_db)
    f_adotada=freq_adotada_atenuacao(a_db)
    rows.append({'Condição':nome,'Ideal [Hz]':f_ideal,'Adotado [Hz]':f_adotada,
                 'Erro relativo ao ideal [%]':100*(f_adotada/f_ideal-1)})
resultados=pd.DataFrame(rows)
display(resultados.round(6))

FC_ADOTADA=rows[0]['Adotado [Hz]']
F_M10=rows[1]['Adotado [Hz]']
F_M34=rows[2]['Adotado [Hz]']
print(f'Erro absoluto de f_c em relação a 50 Hz: {abs(FC_ADOTADA-FC)/FC*100:.4f}%')
print(f'Razão f(−34 dB)/f(−10 dB) = {F_M34/F_M10:.6f}')
print(f'Inclinação assintótica = {-20*ORDEM} dB/década = {-20*ORDEM*np.log10(2):.4f} dB/oitava')

## 11. Diagrama de Bode — magnitude em decibéis

Traçam-se as curvas do Butterworth ideal e da montagem que usa suas associações de capacitores. A referência $0\,\mathrm{dB}$ relativa à banda passante corresponde a $20\log_{10}4=12{,}0412\,\mathrm{dB}$ no gráfico absoluto.

In [ ]:
f_plot=np.geomspace(1,1000,1600)
G0=20*np.log10(GANHO)
plt.figure(figsize=(11,6))
plt.semilogx(f_plot,db(h_ideal(f_plot)),label='Butterworth ideal (50 Hz)',lw=2)
plt.semilogx(f_plot,db(h_adotado(f_plot)),label='Capacitores disponíveis',lw=2,ls='--')
plt.axvline(FC,ls=':',c='0.35',label='50 Hz (projeto)')
plt.axvline(FC_ADOTADA,ls='-.',c='0.5',label=f'{FC_ADOTADA:.2f} Hz (adotado)')
for atten in [-3.01029995664,-10,-34]:
    plt.axhline(G0+atten,ls=':',lw=0.8,c='0.65')
plt.xlabel('Frequência (Hz)')
plt.ylabel('Magnitude (dB)')
plt.title('Bode de magnitude — filtro Butterworth de 4ª ordem, K = 4')
plt.ylim(-90,16)
plt.grid(True,which='both',alpha=0.35)
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

## 12. Diagrama de Bode — fase

Para cada segundo grau, a fase é o negativo do argumento do denominador. Para o sistema completo, somam-se as fases dos dois estágios. O estágio não inversor de ganho positivo não acrescenta fase no modelo ideal.

In [ ]:
plt.figure(figsize=(11,5))
phase_ideal=np.rad2deg(np.unwrap(np.angle(h_ideal(f_plot))))
phase_adotada=np.rad2deg(np.unwrap(np.angle(h_adotado(f_plot))))
plt.semilogx(f_plot,phase_ideal,label='Butterworth ideal',lw=2)
plt.semilogx(f_plot,phase_adotada,label='Capacitores disponíveis',lw=2,ls='--')
plt.axvline(FC,ls=':',c='0.4')
plt.xlabel('Frequência (Hz)')
plt.ylabel('Fase (graus)')
plt.title('Bode de fase — dois estágios de 2ª ordem')
plt.ylim(-380,10)
plt.grid(True,which='both',alpha=0.35)
plt.legend()
plt.tight_layout()
plt.show()

## 13. Tabela de previsões para osciloscópio

O roteiro solicita inicialmente 1 Hz, seguido dos múltiplos de $f_c$: $0{,}25$, $0{,}5$, $0{,}75$, $1$, $1{,}25$, $1{,}6$ e $2{,}5$.

$$V_{o,pp}=|H(j2\pi f)|\,V_{i,pp}.$$

Exibimos tanto a saída do filtro em ganho unitário (dois seguidores) quanto a saída teórica com o ganho adicional $K=4$.

**Limitação física:** para $V_{i,pp}=5\,\mathrm{Vpp}$ e $K=4$, espera-se $20\,\mathrm{Vpp}$ em baixas frequências. Essa excursão não é realizável sem recorte com um amplificador comum alimentado a $\pm10\,\mathrm{V}$. Para validar os filtros em ganho unitário, use a coluna correspondente; para testar o circuito completo, reduza a amplitude de entrada e confira os limites reais do amplificador.

In [ ]:
multiplicadores=[0.25,0.5,0.75,1.0,1.25,1.6,2.5]
frequencias=[1.0]+[FC*k for k in multiplicadores]
registro=[]
for f in frequencias:
    ganho_ideal=abs(h_ideal(f))
    ganho_real=abs(h_adotado(f))
    registro.append({
        'Frequência [Hz]':f,
        'Atenuação relativa ideal [dB]':20*np.log10(ganho_ideal/GANHO),
        'Atenuação relativa adotada [dB]':20*np.log10(ganho_real/GANHO),
        'Saída adotada K=1 [Vpp]':VIN_PP*(ganho_real/GANHO),
        'Saída ideal K=4 [Vpp]':VIN_PP*ganho_ideal,
        'Saída adotada K=4 [Vpp]':VIN_PP*ganho_real,
    })
display(pd.DataFrame(registro).round(5))

## 14. Resumo dos componentes e resultados

Os números abaixo resultam **do dimensionamento e das associações escolhidas**, sem aferição física de tolerâncias. Para a análise pós-laboratorial será necessário substituir as capacitâncias e resistências nominais pelos valores medidos antes da montagem.

In [ ]:
print('ESPECIFICAÇÕES')
print(f'f_c nominal: {FC:.3f} Hz | ganho total: {GANHO:.1f} | ordem: {ORDEM}')
print('\nMONTAGEM — CAPACITORES DISPONÍVEIS')
for nome,r1,r2,c1,c2 in etapas:
    print(f'{nome}: R1=R2={r1/1e3:.2f} kΩ; C1={c1*1e9:.5f} nF; C2={c2*1e9:.5f} nF')
print('\nMÍNIMO PARA ESTA COMBINAÇÃO')
print('22 nF: 5 unidades | 47 nF: 2 unidades | 100 nF: 1 unidade')
print('\nRESULTADOS NOMINAIS DA MONTAGEM')
print(f'Frequência de corte: {FC_ADOTADA:.6f} Hz')
print(f'Frequência em −10 dB: {F_M10:.6f} Hz')
print(f'Frequência em −34 dB: {F_M34:.6f} Hz')
print(f'Erro de corte: {abs(FC_ADOTADA-FC)/FC*100:.4f}%')
print('Ganho de 4 V/V realizado em estágio separado (Rf=30 kΩ, Rg=10 kΩ).')

## Referências e observações

- **UFPE — Departamento de Eletrônica e Sistemas:** *EL216 — Circuitos 2, 5ª Prática de Laboratório*, Prof. Hermano Cabral, 2022. Fonte do roteiro, das especificações e da topologia de filtro.
- **UFPE — Circuitos 2:** slides *Filtro Butterworth*, Prof. Hermano Cabral, 7 abr. 2022. Fonte dos polos normalizados, da síntese e dos procedimentos de escalonamento.
- **Texas Instruments:** *Analysis of the Sallen-Key Architecture*, application report SLOA024B: https://www.ti.com/lit/an/sloa024b/sloa024b.pdf
- **Texas Instruments:** *Active Low-Pass Filter Design*, SLOA049D: https://www.ti.com/lit/an/sloa049d/sloa049d.pdf

**Notas:** o projeto assume amplificadores ideais em regime linear, componentes sem parasitas e cargas com impedância suficientemente alta. A substituição de componentes comerciais por associações altera ligeiramente polos e frequência de corte; os cálculos do notebook incorporam essa alteração. A análise de documentos de terceiros foi deliberadamente excluída.